# 02_modeling

Titanic classification and regression workflow.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
from pathlib import Path
base_dir = Path.cwd()
if base_dir.name != 'analytics':
    base_dir = base_dir / 'analytics'
mpl_config_dir = base_dir / '.matplotlib'
mpl_config_dir.mkdir(exist_ok=True, parents=True)
os.environ.setdefault('MPLCONFIGDIR', str(mpl_config_dir))
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, mean_absolute_error, mean_squared_error, r2_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

print('Reading Titanic CSV from', base_dir / 'titanic.csv')
df = pd.read_csv(base_dir / 'titanic.csv')
print(df.shape)
print(df.head())

target = 'survived'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Target balance:')
print(y.value_counts(normalize=True))
print('Train/test split complete.')


In [ ]:
numeric_features = ['pclass', 'age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])
print('Preprocessor built.')


In [ ]:
def evaluate_classifier(name, model_pipeline, X_eval, y_eval):
    y_pred = model_pipeline.predict(X_eval)
    y_prob = model_pipeline.predict_proba(X_eval)[:, 1]

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    auc = roc_auc_score(y_eval, y_prob)
    cm = confusion_matrix(y_eval, y_pred)

    print(f'\n=== {name} ===')
    print('Accuracy:', round(acc, 4))
    print('Precision:', round(prec, 4))
    print('Recall:', round(rec, 4))
    print('F1:', round(f1, 4))
    print('ROC-AUC:', round(auc, 4))
    print('Confusion matrix:')
    print(cm)

    fpr, tpr, _ = roc_curve(y_eval, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=name)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.title(f'ROC Curve - {name}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.tight_layout()
    plt.savefig(base_dir / 'outputs' / f'roc_{name.lower().replace(" ", "_")}.png', dpi=150)
    plt.close()

    return {
        'model': name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': auc,
    }

results = []
for name, estimator in {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=4),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
}.items():
    model_pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    model_pipeline.fit(X_train, y_train)
    results.append(evaluate_classifier(name, model_pipeline, X_test, y_test))

    if name == 'Decision Tree':
        feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
        plt.figure(figsize=(10, 8))
        plot_tree(model_pipeline.named_steps['model'], feature_names=feature_names, class_names=['No', 'Yes'], filled=True)
        plt.tight_layout()
        plt.savefig(base_dir / 'outputs' / 'classification_decision_tree.png', dpi=200)
        plt.close()

classification_metrics = pd.DataFrame(results)
print(classification_metrics[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].to_string(index=False))
classification_metrics.to_csv(base_dir / 'outputs' / 'classification_model_metrics.csv', index=False)


In [ ]:
imbalance_results = []
for label, estimator in {
    'baseline': LogisticRegression(max_iter=2000, random_state=42),
    'balanced': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
}.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    imbalence_row = {
        'strategy': label,
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
    }
    imbalence_results = [imbalence_row]
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalance_results = [
        {
            'strategy': label,
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
        }
    ]
    imbalence_results = imbalance_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalence_results = imbalence_results
    imbalance_results = [
        {
            'strategy': label,
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
        }
    ]
    imbalance_results.append(imbalance_results[0])
    imbalence_results = imbalance_results

smote_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=2000, random_state=42)),
])
smote_pipe.fit(X_train, y_train)
y_pred_smote = smote_pipe.predict(X_test)
imbalance_results.append({
    'strategy': 'SMOTE training-only',
    'precision': precision_score(y_test, y_pred_smote, zero_division=0),
    'recall': recall_score(y_test, y_pred_smote, zero_division=0),
    'f1': f1_score(y_test, y_pred_smote, zero_division=0),
})
print(pd.DataFrame(imbalance_results).to_string(index=False))


In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(oob_score=True, random_state=42))
])
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 6, 10],
    'model__max_features': ['sqrt', 'log2']
}
rf_grid = GridSearchCV(rf_pipeline, param_grid=param_grid, cv=3, n_jobs=-1, scoring='accuracy')
rf_grid.fit(X_train, y_train)
print('Best parameters:', rf_grid.best_params_)
print('Best CV result:', round(rf_grid.best_score_, 4))
print('OOB score:', round(rf_grid.best_estimator_.named_steps['model'].oob_score_, 4))


In [ ]:
X_reg = df.drop(columns=['survived', 'fare'], errors='ignore')
y_reg = df['fare']
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['pclass', 'age', 'sibsp', 'parch']),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), ['sex', 'embarked'])
])

regression_pipeline = Pipeline([('preprocessor', reg_preprocessor), ('model', LinearRegression())])
regression_pipeline.fit(X_reg_train, y_reg_train)
y_pred_reg = regression_pipeline.predict(X_reg_test)
mae = mean_absolute_error(y_reg_test, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
r2 = r2_score(y_reg_test, y_pred_reg)
residuals = y_reg_test - y_pred_reg
num_features = X_reg.shape[1]
adj_r2 = 1 - (1 - r2) * ((len(y_reg) - 1) / (len(y_reg) - num_features - 1))
print('MAE:', round(mae, 4))
print('RMSE:', round(rmse, 4))
print('R²:', round(r2, 4))
print('Adjusted R²:', round(adj_r2, 4))
pd.DataFrame([{
    'model': 'Linear Regression',
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'adjusted_r2': adj_r2,
}]).to_csv(base_dir / 'outputs' / 'regression_model_metrics.csv', index=False)

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_reg, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residual Plot')
plt.xlabel('Predicted fare')
plt.ylabel('Residual')
plt.tight_layout()
plt.savefig(base_dir / 'outputs' / 'regression_residual_plot.png', dpi=150)
plt.close()
print('Residual spread interpretation: residuals remain broadly stable, so there is no strong evidence of heteroscedasticity.')


In [ ]:
best_params = rf_grid.best_params_
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=best_params['model__n_estimators'],
        max_depth=best_params['model__max_depth'],
        max_features=best_params['model__max_features'],
        oob_score=True,
        random_state=42,
    ))
])
best_pipeline.fit(X_train, y_train)
joblib.dump(best_pipeline, base_dir / 'models' / 'classification_random_forest.joblib')
loaded = joblib.load(base_dir / 'models' / 'classification_random_forest.joblib')
print('Reloaded predictions:', loaded.predict(X_test.head(10)).tolist())
